# US Aviation — Data Preprocessing & Cleaning

This notebook cleans the raw 3-million-row flights dataset and produces:
- `cleaned_flights.csv` — fully cleaned dataset (kept in Drive)
- `flights_model_ready_3class.csv` — feature-engineered, label-encoded dataset ready for ML

Pipeline steps:
1. Mount Google Drive & set paths
2. Install / import dependencies
3. Compute Winsorization bounds (sampled pass)
4. Chunked cleaning pass → write cleaned CSV
5. Generate & display cleaning report
6. Feature engineering → write model-ready CSV

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import subprocess, sys

def _has_gpu():
    try:
        subprocess.run(["nvidia-smi"], check=True, capture_output=True)
        return True
    except Exception:
        return False

USE_GPU = _has_gpu()

if USE_GPU:
    print("GPU detected — installing RAPIDS cuDF…")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet",
         "--extra-index-url", "https://pypi.nvidia.com",
         "cudf-cu12", "cupy-cuda12x"],
        check=True
    )
    print("cuDF installed successfully.")
else:
    print("No GPU detected — will use standard pandas (CPU). "
          "Switch to a T4 runtime for GPU acceleration.")

GPU detected — installing RAPIDS cuDF…
cuDF installed successfully.


In [ ]:
# cuDF as drop-in for pandas when GPU is available; real pandas otherwise.
# All subsequent code uses `pd` — no changes needed downstream.
import json
from dataclasses import dataclass, asdict

import numpy as _np          # always real numpy  (used for np.select in string helpers)
import pandas as _pd         # always real pandas (used for CSV I/O and string helpers)

if USE_GPU:
    import cudf as pd
    import cupy as cp
    print(f"Running with cuDF {pd.__version__} on GPU.")
else:
    import pandas as pd
    print(f"Running with pandas {pd.__version__} on CPU.")

Running with cuDF 26.02.01 on GPU.


In [ ]:
from pathlib import Path


RAW_INPUT    = Path("/content/drive/MyDrive/aviation/flights_sample_3m.csv")
CLEANED_DATA = Path("/content/drive/MyDrive/aviation/cleaned_flights.csv")
MODEL_READY  = Path("/content/drive/MyDrive/aviation/flights_model_ready_3class.csv")
REPORT_JSON  = Path("/content/drive/MyDrive/aviation/cleaning_report.json")

CLEANED_DATA.parent.mkdir(parents=True, exist_ok=True)

print("Paths configured:")
print(f"  RAW_INPUT   : {RAW_INPUT}")
print(f"  CLEANED_DATA: {CLEANED_DATA}")
print(f"  MODEL_READY : {MODEL_READY}")
print(f"  REPORT_JSON : {REPORT_JSON}")

Paths configured:
  RAW_INPUT   : /content/drive/MyDrive/aviation/flights_sample_3m.csv
  CLEANED_DATA: /content/drive/MyDrive/aviation/cleaned_flights.csv
  MODEL_READY : /content/drive/MyDrive/aviation/flights_model_ready_3class.csv
  REPORT_JSON : /content/drive/MyDrive/aviation/cleaning_report.json


## Constants & Column Lists

In [ ]:
CHUNKSIZE = 200_000

# EDA mode: keep cancelled & diverted rows (set True to drop them for ML-only output)
DROP_CANCELLED = False
DROP_DIVERTED  = False

DROP_COLS = ["AIRLINE_DOT", "ORIGIN_CITY", "DEST_CITY"]

TIME_COLS = [
    "CRS_DEP_TIME", "DEP_TIME", "WHEELS_OFF",
    "WHEELS_ON", "CRS_ARR_TIME", "ARR_TIME",
]

NUMERIC_COLS = [
    "DOT_CODE", "FL_NUMBER", "CRS_DEP_TIME", "DEP_TIME", "DEP_DELAY",
    "TAXI_OUT", "WHEELS_OFF", "WHEELS_ON", "TAXI_IN", "CRS_ARR_TIME",
    "ARR_TIME", "ARR_DELAY", "CANCELLED", "DIVERTED", "CRS_ELAPSED_TIME",
    "ELAPSED_TIME", "AIR_TIME", "DISTANCE",
    "DELAY_DUE_CARRIER", "DELAY_DUE_WEATHER", "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY", "DELAY_DUE_LATE_AIRCRAFT",
]

DELAY_COMPONENT_COLS = [
    "DELAY_DUE_CARRIER", "DELAY_DUE_WEATHER", "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY", "DELAY_DUE_LATE_AIRCRAFT",
]

WINSOR_COLS      = ["DEP_DELAY", "ARR_DELAY", "TAXI_OUT", "TAXI_IN", "AIR_TIME", "ELAPSED_TIME"]
NONNEGATIVE_COLS = ["TAXI_OUT", "TAXI_IN", "AIR_TIME", "ELAPSED_TIME"]

PLACEHOLDER_STRINGS = ["", " ", "NA", "N/A", "NULL", "NONE", "UNKNOWN", "nan", "NaN", "null"]

CANCELLED_RAW_COLS = [
    "DEP_TIME", "DEP_DELAY", "TAXI_OUT", "WHEELS_OFF",
    "WHEELS_ON", "TAXI_IN", "ARR_TIME", "ARR_DELAY",
    "ELAPSED_TIME", "AIR_TIME",
    "DELAY_DUE_CARRIER", "DELAY_DUE_WEATHER", "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY", "DELAY_DUE_LATE_AIRCRAFT",
]

CANCELLED_DERIVED_COLS = [
    "DEP_TIME_MIN", "WHEELS_OFF_MIN", "WHEELS_ON_MIN", "ARR_TIME_MIN",
    "DEP_DT", "ARR_DT", "ACTUAL_BLOCK_MINS",
    "DEP_DELAY_CLEAN", "ARR_DELAY_CLEAN", "TAXI_OUT_CLEAN", "TAXI_IN_CLEAN",
    "AIR_TIME_CLEAN", "ELAPSED_TIME_CLEAN",
]

DIVERTED_ARRIVAL_COLS = [
    "WHEELS_ON", "TAXI_IN", "ARR_TIME", "ARR_DELAY",
    "ELAPSED_TIME", "AIR_TIME",
    "DELAY_DUE_CARRIER", "DELAY_DUE_WEATHER", "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY", "DELAY_DUE_LATE_AIRCRAFT",
    "WHEELS_ON_MIN", "ARR_TIME_MIN", "ARR_DT", "ACTUAL_BLOCK_MINS",
    "ARR_DELAY_CLEAN", "TAXI_IN_CLEAN", "AIR_TIME_CLEAN", "ELAPSED_TIME_CLEAN",
]

print("Constants defined.")

Constants defined.


## Helper Functions (cuDF-Compatible)

All helpers use fully vectorized operations — no Python-level `.apply()` loops.
This is required for cuDF (which doesn't support arbitrary Python apply) and also
makes the pandas CPU path significantly faster on 3M rows.

Key design choices:
- `pd`  = cudf or pandas depending on runtime (set in Cell 5)
- `_pd` = always real pandas — used for string bucketing via numpy.select
- `_np` = always real numpy — used for np.select (host-side array ops)

In [ ]:
# ── String normalization ───────────────────────────────────────────────────────

def normalize_string_placeholders(series):
    """Strip whitespace and null out common placeholder strings."""
    s = series.astype("string").str.strip()
    # .isin() works on both pandas StringDtype and cuDF string series
    return s.where(~s.isin(PLACEHOLDER_STRINGS), other=None)


# ── HHMM → minutes since midnight (vectorized, replaces scalar apply) ─────────

def hhmm_to_minutes_vec(series):
    """
    Convert BTS HHMM integer column (e.g. 1435 → 863 min) to minutes since midnight.
    Fully vectorized: works with pandas and cuDF without Python-level loops.
    """
    s = pd.to_numeric(series, errors="coerce").astype("float64")
    s = s.where(s != 2400, other=0.0)      # 2400 → midnight
    s = s.where(s >= 0)                     # negative values → NaN
    hh = s // 100
    mm = s % 100
    invalid = (hh > 24) | (mm > 59) | ((hh == 24) & (mm != 0))
    total = (hh * 60 + mm).where(~invalid).where(s.notna())
    return total.astype("Int64")


# ── Datetime construction ──────────────────────────────────────────────────────

def build_event_datetime(flight_date, minutes, ref_minutes=None):
    """
    Build a datetime from a date column + minutes-since-midnight.
    ref_minutes: if provided, adds 1 day when minutes < ref_minutes (midnight rollover).
    Runs entirely on real pandas (cuDF lacks to_timedelta and errors='coerce' support);
    result is converted back to a cuDF Series when USE_GPU=True.
    """
    _fd   = flight_date.to_pandas()   if USE_GPU else flight_date
    _mins = minutes.to_pandas()        if USE_GPU else minutes
    _ref  = ref_minutes.to_pandas()    if (USE_GPU and ref_minutes is not None) else ref_minutes

    dt   = _pd.to_datetime(_fd, errors="coerce")
    mins = _mins.astype("Int64")
    out  = dt + _pd.to_timedelta(mins.fillna(0), unit="m")
    out  = out.where(~mins.isna(), other=_pd.NaT)

    if _ref is not None:
        ref      = _ref.astype("Int64")
        rollover = (~mins.isna()) & (~ref.isna()) & (mins < ref)
        out      = out + _pd.to_timedelta(rollover.astype(int) * 1440, unit="m")

    return pd.Series(out) if USE_GPU else out


# ── City / state string splitting ─────────────────────────────────────────────

def clean_city_name(series):
    s = normalize_string_placeholders(series)
    return s.str.title()


def split_city_state(s):
    """Split 'City, ST' into title-case city and upper-case state series."""
    s     = normalize_string_placeholders(s)
    parts = s.str.split(",", n=1, expand=True)   # returns DataFrame in both pandas & cuDF
    city  = parts[0].str.strip().str.title().astype("string")
    if parts.shape[1] > 1:
        state = parts[1].str.strip().str.upper().astype("string")
    else:
        # No comma found — return all-null series with same length
        state = city.where(city.notna() & False)   # all-null, preserves dtype
    return city, state


# ── String bucket helpers (use numpy.select on host, then return pandas Series) ─

def dep_time_bucket_vec(minutes_series):
    """
    Convert CRS_DEP_TIME_MIN to a named time-of-day bucket.
    Uses numpy.select on the host (CPU) side since string categoricals
    are not a cuDF bottleneck and np.select is simpler than cuDF string ops.
    """
    arr = pd.to_numeric(minutes_series, errors="coerce")
    # Pull to real numpy for np.select
    arr_np = (arr.to_pandas().values if USE_GPU else arr.values).astype(float)
    hour      = arr_np // 60
    nan_mask  = _np.isnan(arr_np)
    result = _np.select(
        [(hour >= 5)  & (hour < 9),
         (hour >= 9)  & (hour < 12),
         (hour >= 12) & (hour < 17),
         (hour >= 17) & (hour < 21)],
        ["Early Morning", "Morning", "Afternoon", "Evening"],
        default="Night"
    ).astype(object)
    result[nan_mask] = None
    return _pd.Series(result)   # return real pandas Series (safe for CSV writing)


def season_vec(month_series):
    """Vectorized month → meteorological season string."""
    arr = pd.to_numeric(month_series, errors="coerce")
    arr_np   = (arr.to_pandas().values if USE_GPU else arr.values).astype(float)
    nan_mask = _np.isnan(arr_np)
    result = _np.select(
        [_np.isin(arr_np, [12, 1, 2]),
         _np.isin(arr_np, [3, 4, 5]),
         _np.isin(arr_np, [6, 7, 8])],
        ["Winter", "Spring", "Summer"],
        default="Fall"
    ).astype(object)
    result[nan_mask] = None
    return _pd.Series(result)


# ── Delay class label (vectorized, replaces scalar apply) ─────────────────────

def delay_class_3_vec(series):
    """
    0 = on-time/minor (≤ 15 min), 1 = moderate (16–60 min), 2 = severe (> 60 min).
    Trick: (dep_delay > 15) + (dep_delay > 60) gives 0 / 1 / 2 directly.
    Works on both pandas and cuDF numeric series.
    """
    s      = pd.to_numeric(series, errors="coerce")
    result = (s > 15).astype("int8") + (s > 60).astype("int8")
    return result.where(s.notna()).astype("Int64")


print("Helper functions defined (fully vectorized, cuDF-compatible).")

Helper functions defined (fully vectorized, cuDF-compatible).


## Step 1 — Compute Winsorization Bounds (Sampled Pass)

A single fast sampling pass over the raw CSV computes IQR-based clip bounds
for delay and time columns. We always use real pandas here since cuDF's chunked
CSV reading is not needed — only ~2% of each chunk is sampled.

In [ ]:
def compute_winsor_bounds(input_path, chunksize=CHUNKSIZE, sample_target=200_000):
    rng            = _np.random.default_rng(7)
    samples        = {col: [] for col in WINSOR_COLS}
    per_col_target = max(20_000, sample_target // max(1, len(WINSOR_COLS)))

    # Always use real pandas for CSV reading (chunked I/O)
    for chunk in _pd.read_csv(input_path, chunksize=chunksize):
        for col in WINSOR_COLS:
            if col not in chunk.columns:
                continue
            x         = _pd.to_numeric(chunk[col], errors="coerce").dropna().to_numpy(dtype=float)
            current_n = sum(arr.size for arr in samples[col])
            remaining = per_col_target - current_n
            if remaining <= 0 or x.size == 0:
                continue
            take   = min(remaining, max(1000, int(0.02 * x.size)))
            chosen = x if take >= x.size else x[rng.choice(x.size, size=take, replace=False)]
            samples[col].append(chosen)

    bounds = {}
    for col, arrays in samples.items():
        if not arrays:
            continue
        x        = _np.concatenate(arrays)
        q1, q3   = float(_np.percentile(x, 25)), float(_np.percentile(x, 75))
        iqr      = q3 - q1
        if iqr == 0:
            lower, upper = q1, q3
        else:
            lower, upper = q1 - 3.0 * iqr, q3 + 3.0 * iqr
        if col in NONNEGATIVE_COLS:
            lower = max(0.0, lower)
        bounds[col] = {"lower": lower, "upper": upper}

    return bounds


print("Computing Winsorization bounds (sampling pass)…")
winsor_bounds = compute_winsor_bounds(RAW_INPUT)

print("\nWinsorization bounds:")
for col, b in winsor_bounds.items():
    print(f"  {col:25s}  lower={b['lower']:8.1f}  upper={b['upper']:8.1f}")

Computing Winsorization bounds (sampling pass)…

Winsorization bounds:
  DEP_DELAY                  lower=   -42.0  upper=    42.0
  ARR_DELAY                  lower=   -81.0  upper=    73.0
  TAXI_OUT                   lower=     0.0  upper=    43.0
  TAXI_IN                    lower=     0.0  upper=    24.0
  AIR_TIME                   lower=     0.0  upper=   385.0
  ELAPSED_TIME               lower=     0.0  upper=   424.0


## Fig. 5 — Boxplots BEFORE Winsorization

Shows the raw distribution of key numerical columns before outlier treatment.
Extreme values in DEP_DELAY, ARR_DELAY, TAXI_OUT etc. are clearly visible.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("Reading raw sample for pre-Winsorization boxplots…")
raw_parts = []
for chunk in _pd.read_csv(RAW_INPUT, chunksize=CHUNKSIZE, usecols=WINSOR_COLS):
    raw_parts.append(chunk)
    if sum(len(p) for p in raw_parts) >= 400_000:
        break
raw_sample = _pd.concat(raw_parts, ignore_index=True)

fig, axes = plt.subplots(1, len(WINSOR_COLS), figsize=(18, 4))
for ax, col in zip(axes, WINSOR_COLS):
    data = raw_sample[col].dropna()
    ax.boxplot(data, vert=True, patch_artist=True,
               boxprops=dict(facecolor="steelblue", color="navy"),
               medianprops=dict(color="white", linewidth=2))
    ax.set_title(f"Outliers in {col}", fontsize=9)
    ax.set_xlabel(col, fontsize=8)

plt.suptitle("BEFORE PREPROCESSING", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "boxplots_before_winsor.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()
del raw_sample, raw_parts
print("Saved → boxplots_before_winsor.png")

## Step 2 — Per-Chunk Cleaning Function

`process_chunk()` receives a pandas DataFrame (read from CSV), optionally converts
it to a cuDF DataFrame for GPU-accelerated processing, applies all 15 cleaning
steps, then converts the result back to pandas before returning — so CSV writing
always stays on the CPU side.

In [ ]:
def null_operational_fields_for_cancelled(df):
    if "CANCELLED" not in df.columns:
        return df
    mask = df["CANCELLED"] == 1
    for col in CANCELLED_RAW_COLS + CANCELLED_DERIVED_COLS:
        if col in df.columns:
            df.loc[mask, col] = None
    return df


def null_arrival_fields_for_diverted(df):
    if "DIVERTED" not in df.columns:
        return df
    cancelled = df["CANCELLED"] if "CANCELLED" in df.columns else 0
    mask = (df["DIVERTED"] == 1) & (cancelled == 0)
    for col in DIVERTED_ARRIVAL_COLS:
        if col in df.columns:
            df.loc[mask, col] = None
    return df


def process_chunk(pandas_chunk, winsor_bounds):
    """
    Clean one chunk.
    - Converts to cuDF DataFrame if USE_GPU=True (GPU processing).
    - All operations use vectorized `pd` API (cudf or pandas).
    - String-bucket columns (dep_time_bucket, season) stay as real pandas Series.
    - Returns a real pandas DataFrame ready for CSV writing.
    """
    # ── Convert to cuDF if GPU available ──────────────────────────────────────
    if USE_GPU:
        # cudf.to_datetime does not support errors='coerce' on a Series —
        # parse all date columns with real pandas before GPU transfer.
        _tmp = pandas_chunk.copy()
        if "FL_DATE" in _tmp.columns:
            _tmp["FL_DATE"] = _pd.to_datetime(_tmp["FL_DATE"], errors="coerce")
        df = pd.from_pandas(_tmp)
    else:
        df = pandas_chunk.copy()

    # 1. Normalize string placeholders
    for col in df.columns:
        if df[col].dtype == object or str(df[col].dtype).startswith("string"):
            df[col] = normalize_string_placeholders(df[col])

    # 2. Parse flight date (CPU path only — GPU path already parsed above)
    if "FL_DATE" in df.columns and not str(df["FL_DATE"].dtype).startswith("datetime"):
        df["FL_DATE"] = pd.to_datetime(df["FL_DATE"], errors="coerce")

    # 3. Coerce numeric columns
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 4. Binary flags — default 0 when missing
    for col in ["CANCELLED", "DIVERTED"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int32")

    # 5. Cancellation code: blank for non-cancelled rows
    if "CANCELLATION_CODE" in df.columns:
        df["CANCELLATION_CODE"] = normalize_string_placeholders(df["CANCELLATION_CODE"])
        if "CANCELLED" in df.columns:
            df.loc[df["CANCELLED"] == 0, "CANCELLATION_CODE"] = None

    # 6. Delay components: fill 0 for completed flights
    for col in DELAY_COMPONENT_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            if "CANCELLED" in df.columns and "DIVERTED" in df.columns:
                mask = (df["CANCELLED"] == 0) & (df["DIVERTED"] == 0) & df[col].isna()
                df.loc[mask, col] = 0.0

    # 7. City / state splitting
    if "ORIGIN_CITY" in df.columns:
        df["ORIGIN_CITY_NAME"], df["ORIGIN_STATE"] = split_city_state(df["ORIGIN_CITY"])
    elif "ORIGIN_CITY_NAME" in df.columns:
        df["ORIGIN_CITY_NAME"] = clean_city_name(df["ORIGIN_CITY_NAME"])

    if "DEST_CITY" in df.columns:
        df["DEST_CITY_NAME"], df["DEST_STATE"] = split_city_state(df["DEST_CITY"])
    elif "DEST_CITY_NAME" in df.columns:
        df["DEST_CITY_NAME"] = clean_city_name(df["DEST_CITY_NAME"])

    # 8. HHMM → minutes since midnight (vectorized, no apply)
    for col in TIME_COLS:
        if col in df.columns:
            df[f"{col}_MIN"] = hhmm_to_minutes_vec(df[col])

    # 9. Build datetime columns (handles midnight rollover)
    if "FL_DATE" in df.columns and "CRS_DEP_TIME_MIN" in df.columns:
        df["CRS_DEP_DT"] = build_event_datetime(df["FL_DATE"], df["CRS_DEP_TIME_MIN"])
    if "FL_DATE" in df.columns and "CRS_ARR_TIME_MIN" in df.columns and "CRS_DEP_TIME_MIN" in df.columns:
        df["CRS_ARR_DT"] = build_event_datetime(df["FL_DATE"], df["CRS_ARR_TIME_MIN"], df["CRS_DEP_TIME_MIN"])
    if "FL_DATE" in df.columns and "DEP_TIME_MIN" in df.columns:
        df["DEP_DT"] = build_event_datetime(df["FL_DATE"], df["DEP_TIME_MIN"])
    if "FL_DATE" in df.columns and "ARR_TIME_MIN" in df.columns and "DEP_TIME_MIN" in df.columns:
        df["ARR_DT"] = build_event_datetime(df["FL_DATE"], df["ARR_TIME_MIN"], df["DEP_TIME_MIN"])

    # 10. Scheduled & actual block times
    if {"CRS_DEP_DT", "CRS_ARR_DT"}.issubset(df.columns):
        df["SCHED_BLOCK_MINS"] = (df["CRS_ARR_DT"] - df["CRS_DEP_DT"]).dt.total_seconds() / 60
    if {"DEP_DT", "ARR_DT"}.issubset(df.columns):
        df["ACTUAL_BLOCK_MINS"] = (df["ARR_DT"] - df["DEP_DT"]).dt.total_seconds() / 60

    # 11. Winsorize delay / time columns
    for col, bounds in winsor_bounds.items():
        if col in df.columns:
            cleaned = pd.to_numeric(df[col], errors="coerce").clip(
                lower=bounds["lower"], upper=bounds["upper"]
            )
            if col in NONNEGATIVE_COLS:
                cleaned = cleaned.clip(lower=0)
            df[f"{col}_CLEAN"] = cleaned

    # 12. Null out fields that are undefined for cancelled / diverted rows
    df = null_operational_fields_for_cancelled(df)
    df = null_arrival_fields_for_diverted(df)

    # ── Convert back to real pandas before row-filtering & writing ────────────
    if USE_GPU:
        df = df.to_pandas()

    # 13. Remove rows with invalid block times
    invalid_sched_removed  = 0
    invalid_actual_removed = 0

    if "SCHED_BLOCK_MINS" in df.columns:
        bad_sched = df["SCHED_BLOCK_MINS"].notna() & (df["SCHED_BLOCK_MINS"] <= 0)
        invalid_sched_removed = int(bad_sched.sum())
        df = df.loc[~bad_sched].copy()

    if "ACTUAL_BLOCK_MINS" in df.columns:
        operational = _pd.Series(True, index=df.index)
        if "CANCELLED" in df.columns:
            operational &= df["CANCELLED"] == 0
        if "DIVERTED" in df.columns:
            operational &= df["DIVERTED"] == 0
        bad_actual = operational & df["ACTUAL_BLOCK_MINS"].notna() & (df["ACTUAL_BLOCK_MINS"] <= 0)
        invalid_actual_removed = int(bad_actual.sum())
        df = df.loc[~bad_actual].copy()

    # 14. Drop unused raw columns
    existing_drop = [c for c in DROP_COLS if c in df.columns]
    if existing_drop:
        df = df.drop(columns=existing_drop)

    # 15. Optionally drop cancelled / diverted rows
    if DROP_CANCELLED and "CANCELLED" in df.columns:
        df = df[df["CANCELLED"] == 0].copy()
    if DROP_DIVERTED and "DIVERTED" in df.columns:
        df = df[df["DIVERTED"] == 0].copy()

    return df, invalid_sched_removed, invalid_actual_removed   # df is always real pandas here


print(f"process_chunk() defined  (GPU mode: {USE_GPU}).")

process_chunk() defined  (GPU mode: True).


## Step 3 — Chunked Cleaning Pass

We always use `pandas.read_csv()` for chunked I/O (cuDF has no native chunked reader).
Each 200k-row chunk is sent to `process_chunk()`, which moves it onto the GPU,
runs all cleaning steps there, then hands back a real pandas DataFrame for writing.

In [ ]:
# ── Pre-scan pass: row count + missing-before stats (real pandas, CPU) ────────
print("Pre-scan: counting rows and missing values…")

missing_before = {}
input_rows     = 0
cancelled_rows = 0
diverted_rows  = 0

for chunk in _pd.read_csv(RAW_INPUT, chunksize=CHUNKSIZE):
    input_rows += len(chunk)
    for k, v in chunk.isna().sum().to_dict().items():
        missing_before[k] = missing_before.get(k, 0) + int(v)
    if "CANCELLED" in chunk.columns:
        c = _pd.to_numeric(chunk["CANCELLED"], errors="coerce").fillna(0).astype(int)
        cancelled_rows += int(c.sum())
    if "DIVERTED" in chunk.columns:
        d = _pd.to_numeric(chunk["DIVERTED"], errors="coerce").fillna(0).astype(int)
        diverted_rows += int(d.sum())

print(f"Total input rows : {input_rows:,}")
print(f"Cancelled rows   : {cancelled_rows:,}")
print(f"Diverted rows    : {diverted_rows:,}")

Pre-scan: counting rows and missing values…
Total input rows : 3,000,000
Cancelled rows   : 79,140
Diverted rows    : 7,056


## Fig. 3 — Missing Values BEFORE Cleaning

Horizontal bar chart of every column that has at least one missing value
in the raw dataset, sorted by count.

In [ ]:
missing_before_series = _pd.Series(
    {k: v for k, v in missing_before.items() if v > 0}
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, max(5, len(missing_before_series) * 0.4)))
missing_before_series.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Columns with Missing Values (Count Only)", fontsize=12)
ax.set_xlabel("Number of Missing Values")
ax.set_ylabel("Columns")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "missing_values_before.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()
print("Saved → missing_values_before.png")
print(f"Columns with missing values: {len(missing_before_series)}")
print(f"Total missing cells        : {missing_before_series.sum():,}")

In [ ]:
# ── Main cleaning pass ────────────────────────────────────────────────────────
print(f"Cleaning pass ({'GPU / cuDF' if USE_GPU else 'CPU / pandas'}) → {CLEANED_DATA.name}…")

if CLEANED_DATA.exists():
    CLEANED_DATA.unlink()

missing_after                     = {}
output_rows                       = 0
invalid_sched_block_rows_removed  = 0
invalid_actual_block_rows_removed = 0
wrote_header                      = False

for i, pandas_chunk in enumerate(_pd.read_csv(RAW_INPUT, chunksize=CHUNKSIZE)):
    # process_chunk handles cuDF conversion internally
    cleaned, bad_sched_n, bad_actual_n = process_chunk(pandas_chunk, winsor_bounds)

    invalid_sched_block_rows_removed  += bad_sched_n
    invalid_actual_block_rows_removed += bad_actual_n
    output_rows                       += len(cleaned)

    for k, v in cleaned.isna().sum().to_dict().items():
        missing_after[k] = missing_after.get(k, 0) + int(v)

    # cleaned is always a real pandas DataFrame here — safe to write directly
    cleaned.to_csv(CLEANED_DATA, index=False, mode="a", header=not wrote_header)
    wrote_header = True

    if (i + 1) % 5 == 0:
        print(f"  chunk {i+1:3d} done  |  rows written so far: {output_rows:,}")

print(f"\nCleaning complete.")
print(f"  Output rows              : {output_rows:,}")
print(f"  Removed (bad sched block): {invalid_sched_block_rows_removed:,}")
print(f"  Removed (bad actual block): {invalid_actual_block_rows_removed:,}")
print(f"  Saved → {CLEANED_DATA}")

Cleaning pass (GPU / cuDF) → cleaned_flights.csv…
  chunk   5 done  |  rows written so far: 999,443
  chunk  10 done  |  rows written so far: 1,998,868
  chunk  15 done  |  rows written so far: 2,998,289

Cleaning complete.
  Output rows              : 2,998,289
  Removed (bad sched block): 705
  Removed (bad actual block): 1,006
  Saved → /content/drive/MyDrive/aviation/cleaned_flights.csv


## Fig. 4 — Missing Values AFTER Cleaning

In [ ]:
missing_after_series = _pd.Series(
    {k: v for k, v in missing_after.items() if v > 0}
).sort_values(ascending=True)

if len(missing_after_series) == 0:
    print("No missing values remain after cleaning.")
else:
    fig, ax = plt.subplots(figsize=(10, max(5, len(missing_after_series) * 0.4)))
    missing_after_series.plot(kind="barh", ax=ax, color="darkorange")
    ax.set_title("Columns with Missing Values After Cleaning (Count Only)", fontsize=12)
    ax.set_xlabel("Number of Missing Values")
    ax.set_ylabel("Columns")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "missing_values_after.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()
    print("Saved → missing_values_after.png")

print(f"Columns with missing values: {len(missing_after_series)}")
print(f"Total missing cells        : {missing_after_series.sum():,}")

## Fig. 6 — Boxplots AFTER Winsorization

Same columns as Fig. 5 — outliers are now clipped within the computed bounds.

In [ ]:
print("Reading cleaned sample for post-Winsorization boxplots…")
clean_parts = []
for chunk in _pd.read_csv(CLEANED_DATA, chunksize=CHUNKSIZE, usecols=WINSOR_COLS):
    clean_parts.append(chunk)
    if sum(len(p) for p in clean_parts) >= 400_000:
        break
clean_sample = _pd.concat(clean_parts, ignore_index=True)

fig, axes = plt.subplots(1, len(WINSOR_COLS), figsize=(18, 4))
for ax, col in zip(axes, WINSOR_COLS):
    data = clean_sample[col].dropna()
    ax.boxplot(data, vert=True, patch_artist=True,
               boxprops=dict(facecolor="steelblue", color="navy"),
               medianprops=dict(color="white", linewidth=2))
    ax.set_title(f"Outliers in {col}", fontsize=9)
    ax.set_xlabel(col, fontsize=8)

plt.suptitle("AFTER PREPROCESSING", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "boxplots_after_winsor.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()
del clean_sample, clean_parts
print("Saved → boxplots_after_winsor.png")

## Step 4 — Save & Display Cleaning Report

In [ ]:
@dataclass
class CleaningReport:
    input_rows: int
    output_rows: int
    dropped_cancelled_rows: int
    dropped_diverted_rows: int
    invalid_sched_block_rows_removed: int
    invalid_actual_block_rows_removed: int
    missing_before: dict
    missing_after: dict
    winsor_bounds: dict
    cancelled_rows: int
    diverted_rows: int


report = CleaningReport(
    input_rows=input_rows,
    output_rows=output_rows,
    dropped_cancelled_rows=cancelled_rows if DROP_CANCELLED else 0,
    dropped_diverted_rows=diverted_rows   if DROP_DIVERTED  else 0,
    invalid_sched_block_rows_removed=invalid_sched_block_rows_removed,
    invalid_actual_block_rows_removed=invalid_actual_block_rows_removed,
    missing_before=missing_before,
    missing_after=missing_after,
    winsor_bounds=winsor_bounds,
    cancelled_rows=cancelled_rows,
    diverted_rows=diverted_rows,
)

REPORT_JSON.write_text(json.dumps(asdict(report), indent=2), encoding="utf-8")
print(f"Report saved → {REPORT_JSON}")

print("\n── Cleaning Summary ──────────────────────────────────")
print(f"  Input rows                      : {report.input_rows:>10,}")
print(f"  Output rows                     : {report.output_rows:>10,}")
print(f"  Rows removed (bad sched block)  : {report.invalid_sched_block_rows_removed:>10,}")
print(f"  Rows removed (bad actual block) : {report.invalid_actual_block_rows_removed:>10,}")
print(f"  Cancelled flights retained      : {report.cancelled_rows:>10,}")
print(f"  Diverted flights retained       : {report.diverted_rows:>10,}")

Report saved → /content/drive/MyDrive/aviation/cleaning_report.json

── Cleaning Summary ──────────────────────────────────
  Input rows                      :  3,000,000
  Output rows                     :  2,998,289
  Rows removed (bad sched block)  :        705
  Rows removed (bad actual block) :      1,006
  Cancelled flights retained      :     79,140
  Diverted flights retained       :      7,056


## Step 5 — Sanity Check on Cleaned Data

In [ ]:
# Always use real pandas for reading back & displaying previews
df_preview = _pd.read_csv(CLEANED_DATA, nrows=5)
print(f"Columns ({len(df_preview.columns)}):")
print(df_preview.columns.tolist())
df_preview.head()

Columns (51):
['FL_DATE', 'AIRLINE', 'AIRLINE_CODE', 'DOT_CODE', 'FL_NUMBER', 'ORIGIN', 'DEST', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE', 'DEST_CITY_NAME', 'DEST_STATE', 'CRS_DEP_TIME_MIN', 'DEP_TIME_MIN', 'WHEELS_OFF_MIN', 'WHEELS_ON_MIN', 'CRS_ARR_TIME_MIN', 'ARR_TIME_MIN', 'CRS_DEP_DT', 'CRS_ARR_DT', 'DEP_DT', 'ARR_DT', 'SCHED_BLOCK_MINS', 'ACTUAL_BLOCK_MINS', 'DEP_DELAY_CLEAN', 'ARR_DELAY_CLEAN', 'TAXI_OUT_CLEAN', 'TAXI_IN_CLEAN', 'AIR_TIME_CLEAN', 'ELAPSED_TIME_CLEAN']


,FL_DATE,AIRLINE,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,...,DEP_DT,ARR_DT,SCHED_BLOCK_MINS,ACTUAL_BLOCK_MINS,DEP_DELAY_CLEAN,ARR_DELAY_CLEAN,TAXI_OUT_CLEAN,TAXI_IN_CLEAN,AIR_TIME_CLEAN,ELAPSED_TIME_CLEAN
0,2019-01-09,United Air Lines Inc.,UA,19977,1562,FLL,EWR,1155,1151.0,-4.0,...,2019-01-09 11:51:00,2019-01-09 14:47:00,186.0,176.0,-4.0,-14.0,19.0,4.0,153.0,176.0
1,2022-11-19,Delta Air Lines Inc.,DL,19790,1149,MSP,SEA,2120,2114.0,-6.0,...,2022-11-19 21:14:00,2022-11-19 23:10:00,115.0,116.0,-6.0,-5.0,9.0,24.0,189.0,236.0
2,2022-07-22,United Air Lines Inc.,UA,19977,459,DEN,MSP,954,1000.0,6.0,...,2022-07-22 10:00:00,2022-07-22 12:52:00,178.0,172.0,6.0,0.0,20.0,5.0,87.0,112.0
3,2023-03-06,Delta Air Lines Inc.,DL,19790,2295,MSP,SFO,1609,1608.0,-1.0,...,2023-03-06 16:08:00,2023-03-06 18:53:00,140.0,165.0,-1.0,24.0,27.0,9.0,249.0,285.0
4,2020-02-23,Spirit Air Lines,NK,20416,407,MCO,DFW,1840,1838.0,-2.0,...,2020-02-23 18:38:00,2020-02-23 20:40:00,121.0,122.0,-2.0,-1.0,15.0,14.0,153.0,182.0


In [ ]:
# Missing-value before vs. after table
mb = _pd.Series(missing_before).rename("before")
ma = _pd.Series(missing_after).rename("after")
mv = _pd.concat([mb, ma], axis=1).fillna(0).astype(int)
mv = mv[mv["before"] > 0].sort_values("before", ascending=False)
print("Missing values before vs. after cleaning (columns with any missing):\n")
print(mv.to_string())

Missing values before vs. after cleaning (columns with any missing):

                          before    after
CANCELLATION_CODE        2920860  2919167
DELAY_DUE_WEATHER        2466137    86177
DELAY_DUE_NAS            2466137    86177
DELAY_DUE_SECURITY       2466137    86177
DELAY_DUE_LATE_AIRCRAFT  2466137    86177
DELAY_DUE_CARRIER        2466137    86177
AIR_TIME                   86198    86179
ELAPSED_TIME               86198    86179
ARR_DELAY                  86198    86179
TAXI_IN                    79944    86179
WHEELS_ON                  79944    86179
ARR_TIME                   79942    86179
TAXI_OUT                   78806    79122
WHEELS_OFF                 78806    79122
DEP_DELAY                  77644    79122
DEP_TIME                   77615    79122
CRS_ELAPSED_TIME              14       14


## Step 6 — Feature Engineering

Builds `flights_model_ready_3class.csv`:
- Filters to non-cancelled, non-diverted flights with valid `DEP_DELAY`
- Adds 3-class delay label (`delay_class_3`: 0 = on-time/minor, 1 = moderate, 2 = severe)
- Adds calendar features: year, month, day, day_of_week, weekend, dep_hour, dep_time_bucket, season, route
- Drops rows missing any required ML column

All label and bucketing functions are vectorized (cuDF-compatible).

In [ ]:
SAFE_BASE_COLS = [
    "FL_DATE", "AIRLINE", "AIRLINE_CODE", "ORIGIN", "DEST",
    "ORIGIN_STATE", "DEST_STATE",
    "CRS_DEP_TIME_MIN", "CRS_ARR_TIME_MIN",
    "CRS_ELAPSED_TIME", "DISTANCE", "SCHED_BLOCK_MINS",
]

REQUIRED_ML_COLS = [
    "FL_DATE", "AIRLINE", "ORIGIN", "DEST",
    "CRS_DEP_TIME_MIN", "CRS_ARR_TIME_MIN",
    "CRS_ELAPSED_TIME", "DISTANCE", "SCHED_BLOCK_MINS", "delay_class_3",
]


def build_model_ready_3class(df):
    """
    Feature engineering for 3-class delay classification.
    Accepts a real pandas DataFrame; does GPU-side computation if USE_GPU.
    Returns a real pandas DataFrame.
    """
    if USE_GPU:
        # Parse FL_DATE with real pandas before GPU transfer (cuDF to_datetime limitation)
        _tmp = df.copy()
        if "FL_DATE" in _tmp.columns:
            _tmp["FL_DATE"] = _pd.to_datetime(_tmp["FL_DATE"], errors="coerce")
        gdf = pd.from_pandas(_tmp)
    else:
        gdf = df.copy()

    # Filter: completed flights with a known departure delay
    if "CANCELLED" in gdf.columns:
        gdf = gdf[gdf["CANCELLED"] == 0].copy()
    if "DIVERTED" in gdf.columns:
        gdf = gdf[gdf["DIVERTED"] == 0].copy()
    gdf = gdf[gdf["DEP_DELAY"].notna()].copy()

    # 3-class delay target (vectorized — no apply)
    gdf["delay_class_3"] = delay_class_3_vec(gdf["DEP_DELAY"])

    # Keep only safe base columns + target
    keep = [c for c in SAFE_BASE_COLS + ["delay_class_3"] if c in gdf.columns]
    gdf  = gdf[keep].copy()

    # Calendar features (FL_DATE already parsed for GPU path; parse here for CPU path)
    if not str(gdf["FL_DATE"].dtype).startswith("datetime"):
        gdf["FL_DATE"] = pd.to_datetime(gdf["FL_DATE"], errors="coerce")
    gdf["year"]        = gdf["FL_DATE"].dt.year
    gdf["month"]       = gdf["FL_DATE"].dt.month
    gdf["day"]         = gdf["FL_DATE"].dt.day
    gdf["day_of_week"] = gdf["FL_DATE"].dt.dayofweek
    gdf["is_weekend"]  = gdf["day_of_week"].isin([5, 6]).astype("int32")
    gdf["dep_hour"]    = (gdf["CRS_DEP_TIME_MIN"] // 60).astype("Int64")
    gdf["arr_hour"]    = (gdf["CRS_ARR_TIME_MIN"] // 60).astype("Int64")
    gdf["route"]       = gdf["ORIGIN"].astype(str) + "_" + gdf["DEST"].astype(str)

    # Convert back to pandas for numpy.select-based string buckets
    if USE_GPU:
        out = gdf.to_pandas()
    else:
        out = gdf

    # String bucket columns (vectorized via numpy.select on host)
    out["dep_time_bucket"] = dep_time_bucket_vec(out["CRS_DEP_TIME_MIN"]).values
    out["season"]          = season_vec(out["month"]).values

    # Drop rows missing any required ML column
    out = out.dropna(subset=[c for c in REQUIRED_ML_COLS if c in out.columns]).copy()
    return out   # always real pandas


print("Feature engineering functions defined.")

Feature engineering functions defined.


## Step 7 — Run Feature Engineering

In [ ]:
print("Loading cleaned data (pandas)…")
df_clean = _pd.read_csv(CLEANED_DATA, low_memory=False)
print(f"Cleaned data shape: {df_clean.shape}")

print(f"Building model-ready dataset ({'GPU' if USE_GPU else 'CPU'})…")
df_model = build_model_ready_3class(df_clean)
print(f"Model-ready shape : {df_model.shape}")

df_model.to_csv(MODEL_READY, index=False)
print(f"Saved → {MODEL_READY}")

# Class distribution
dist   = df_model["delay_class_3"].value_counts().sort_index()
labels = {0: "On-time / minor (≤15 min)", 1: "Moderate (16–60 min)", 2: "Severe (>60 min)"}
print("\nDelay class distribution:")
for cls, count in dist.items():
    pct = count / len(df_model) * 100
    print(f"  Class {int(cls)} — {labels[int(cls)]:30s}: {count:>8,}  ({pct:.1f}%)")

Loading cleaned data (pandas)…
Cleaned data shape: (2998289, 51)
Building model-ready dataset (GPU)…
Model-ready shape : (2912112, 23)
Saved → /content/drive/MyDrive/aviation/flights_model_ready_3class.csv

Delay class distribution:
  Class 0 — On-time / minor (≤15 min)     : 2,401,591  (82.5%)
  Class 1 — Moderate (16–60 min)          :  333,947  (11.5%)
  Class 2 — Severe (>60 min)              :  176,574  (6.1%)


## Step 8 — Final Column Overview

In [ ]:
print("Model-ready columns:\n")
for col in df_model.columns:
    dtype  = df_model[col].dtype
    n_null = df_model[col].isna().sum()
    n_uniq = df_model[col].nunique()
    print(f"  {col:25s}  dtype={str(dtype):12s}  nulls={n_null:>6,}  unique={n_uniq:>7,}")

Model-ready columns:

  FL_DATE                    dtype=datetime64[ns]  nulls=     0  unique=  1,704
  AIRLINE                    dtype=object        nulls=     0  unique=     18
  AIRLINE_CODE               dtype=object        nulls=     0  unique=     18
  ORIGIN                     dtype=object        nulls=     0  unique=    380
  DEST                       dtype=object        nulls=     0  unique=    380
  ORIGIN_STATE               dtype=object        nulls=     0  unique=     54
  DEST_STATE                 dtype=object        nulls=     0  unique=     54
  CRS_DEP_TIME_MIN           dtype=int64         nulls=     0  unique=  1,383
  CRS_ARR_TIME_MIN           dtype=int64         nulls=     0  unique=  1,435
  CRS_ELAPSED_TIME           dtype=float64       nulls=     0  unique=    639
  DISTANCE                   dtype=float64       nulls=     0  unique=  1,726
  SCHED_BLOCK_MINS           dtype=float64       nulls=     0  unique=    885
  delay_class_3              dtype=int64

## Done

Output files saved to Google Drive:

| File | Contents |
|---|---|
| `cleaned_flights.csv` | Fully cleaned dataset (all flights, all columns) |
| `flights_model_ready_3class.csv` | Completed flights only, engineered features, 3-class target |
| `cleaning_report.json` | Row counts, missing-value stats, Winsorization bounds |

**GPU vs CPU summary:**
| Step | GPU (cuDF) | CPU (pandas) |
|---|---|---|
| Type coercion, numeric ops, clip/winsorize | On GPU | On CPU |
| HHMM → minutes (vectorized) | On GPU | On CPU |
| Datetime construction | On GPU | On CPU |
| String bucketing (dep_time_bucket, season) | numpy.select on host | numpy.select on host |
| Chunked CSV read / write | Always real pandas | Always real pandas |

Proceed to model training in the next notebook.